# Conformal Calibration Walk-through

This notebook demonstrates split-conformal calibration and the Adaptive Conformal Inference (ACI) update on synthetic data. We then plot the calibration curve (empirical coverage vs target `1 - alpha`).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from ssl_graph_anomaly.conformal import SplitConformal, AdaptiveConformalInference
from ssl_graph_anomaly.utils import set_global_seed
set_global_seed(17)

## 1. Synthetic non-conformity scores

We draw 5 000 calibration scores from `Exponential(1)` and 5 000 fresh test scores from the same distribution.

In [ ]:
rng = np.random.default_rng(17)
cal = rng.exponential(scale=1.0, size=5000).astype(np.float32)
test = rng.exponential(scale=1.0, size=5000).astype(np.float32)

## 2. Calibrate split-conformal across a grid of `alpha`

In [ ]:
alphas = np.array([0.01, 0.05, 0.10, 0.15, 0.20])
empirical = []
for a in alphas:
    cp = SplitConformal(alpha=float(a))
    cp.calibrate(torch.from_numpy(cal))
    coverage = float((test <= cp.quantile).mean())
    empirical.append(coverage)
empirical = np.array(empirical)
for a, c in zip(alphas, empirical):
    print(f'alpha={a:.2f}  empirical coverage={c:.3f}  target={1.0 - a:.3f}')

## 3. Plot the calibration curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(1.0 - alphas, empirical, 'o-', label='empirical')
ax.plot([0.0, 1.0], [0.0, 1.0], 'k--', label='ideal')
ax.set_xlabel('target coverage 1 - alpha')
ax.set_ylabel('empirical coverage')
ax.set_title('Split-conformal calibration curve')
ax.set_xlim(0.75, 1.01)
ax.set_ylim(0.75, 1.01)
ax.legend()
plt.show()

## 4. Adaptive Conformal Inference under simulated drift

We inject a 5% benign-distribution shift at step 1 000 and let ACI shrink `alpha_t` to recover coverage.

In [ ]:
aci = AdaptiveConformalInference(alpha=0.10, gamma=0.01)
history = []
for step in range(3000):
    covered = (rng.random() > 0.10) if step < 1000 else (rng.random() > 0.15)
    aci.update(covered=bool(covered))
    history.append(aci.alpha_t)
history = np.asarray(history)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history, label='alpha_t under ACI')
ax.axvline(1000, color='red', linestyle='--', label='drift injected')
ax.axhline(0.10, color='gray', linestyle=':', label='nominal alpha')
ax.set_xlabel('time step')
ax.set_ylabel('alpha_t')
ax.set_title('ACI tracking under drift')
ax.legend()
plt.show()

After the drift event at step 1 000, ACI shrinks `alpha_t` to keep the empirical coverage on target.